In [1]:
import pandas as pd

# Read just a sample first — full file is large
df = pd.read_csv('../data/loan.csv', low_memory=False, nrows=5000)

print("Shape (sample):", df.shape)
print("\nColumn list:")
print(df.columns.tolist())

Shape (sample): (5000, 145)

Column list:
['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'open_acc_

In [2]:
import subprocess
# Windows equivalent of wc -l
with open('../data/loan.csv', 'r', encoding='utf-8') as f:
    total_lines = sum(1 for _ in f)
print("Total rows (approx):", total_lines - 1)  # minus header

Total rows (approx): 2260668


In [3]:
key_cols = ['loan_amnt', 'term', 'int_rate', 'grade', 'sub_grade', 'emp_length',
            'home_ownership', 'annual_inc', 'purpose', 'dti', 'delinq_2yrs',
            'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
            'loan_status', 'issue_d', 'addr_state', 'application_type']

available = [c for c in key_cols if c in df.columns]
missing = [c for c in key_cols if c not in df.columns]
print("Available:", available)
print("Missing:", missing)

Available: ['loan_amnt', 'term', 'int_rate', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc', 'purpose', 'dti', 'delinq_2yrs', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'loan_status', 'issue_d', 'addr_state', 'application_type']
Missing: []


In [4]:
print(df['loan_status'].value_counts())

loan_status
Current               4898
Fully Paid              95
Late (31-120 days)       5
In Grace Period          2
Name: count, dtype: int64


In [5]:
import pandas as pd

status_counts = pd.read_csv('../data/loan.csv', usecols=['loan_status'], low_memory=False)['loan_status'].value_counts()
print(status_counts)

loan_status
Fully Paid                                             1041952
Current                                                 919695
Charged Off                                             261655
Late (31-120 days)                                       21897
In Grace Period                                           8952
Late (16-30 days)                                         3737
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     31
Name: count, dtype: int64


In [6]:
import pandas as pd
import numpy as np

# ── 1. Load only the columns we need ──────────────────────────────
cols_to_load = [
    'loan_amnt', 'term', 'int_rate', 'grade', 'sub_grade', 'emp_length',
    'home_ownership', 'annual_inc', 'purpose', 'dti', 'delinq_2yrs',
    'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
    'loan_status', 'issue_d', 'addr_state', 'application_type'
]

df = pd.read_csv('../data/loan.csv', usecols=cols_to_load, low_memory=False)
print("Raw shape:", df.shape)

# ── 2. Filter to resolved loans only ──────────────────────────────
resolved_statuses = ['Fully Paid', 'Charged Off', 'Default']
df = df[df['loan_status'].isin(resolved_statuses)].copy()
print("After filtering to resolved loans:", df.shape)

# ── 3. Create binary target ───────────────────────────────────────
df['default_flag'] = df['loan_status'].apply(
    lambda x: 1 if x in ['Charged Off', 'Default'] else 0
)
print("\nTarget distribution:")
print(df['default_flag'].value_counts(normalize=True))

# ── 4. Clean term (e.g. " 36 months" → 36) ────────────────────────
df['term'] = df['term'].str.strip().str.replace(' months', '').astype(int)

# ── 5. Clean int_rate (e.g. "13.56%" → 13.56) ─────────────────────
# int_rate is often already float in this dataset, but handle both cases
if df['int_rate'].dtype == object:
    df['int_rate'] = df['int_rate'].str.replace('%', '').astype(float)

# ── 6. Clean revol_util (e.g. "45.2%" → 45.2) ─────────────────────
if df['revol_util'].dtype == object:
    df['revol_util'] = df['revol_util'].str.replace('%', '').astype(float)

# ── 7. Clean emp_length into numeric years ────────────────────────
emp_length_map = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
    '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9,
    '10+ years': 10
}
df['emp_length_years'] = df['emp_length'].map(emp_length_map)
# missing emp_length -> flag separately rather than guessing a value
df['emp_length_missing'] = df['emp_length_years'].isna().astype(int)
df['emp_length_years'] = df['emp_length_years'].fillna(-1)  # placeholder, flagged above

# ── 8. Parse issue_d into a real date ─────────────────────────────
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')
df['issue_year'] = df['issue_d'].dt.year
df['issue_quarter'] = df['issue_d'].dt.quarter

# ── 9. Handle missing values in numeric fields ────────────────────
numeric_cols_with_nulls = ['dti', 'revol_util', 'delinq_2yrs', 'open_acc',
                            'pub_rec', 'revol_bal', 'total_acc']
for col in numeric_cols_with_nulls:
    missing_pct = df[col].isna().mean() * 100
    if missing_pct > 0:
        print(f"{col}: {missing_pct:.2f}% missing")

# For dti and revol_util specifically — impute median, flag missingness
for col in ['dti', 'revol_util']:
    df[f'{col}_missing'] = df[col].isna().astype(int)
    df[col] = df[col].fillna(df[col].median())

# Drop any remaining rows with nulls in critical fields (should be very few)
before = len(df)
df = df.dropna(subset=['loan_amnt', 'annual_inc', 'grade', 'purpose'])
print(f"\nDropped {before - len(df)} rows with nulls in critical fields")

# ── 10. Final column selection for export ─────────────────────────
final_cols = [
    'loan_amnt', 'term', 'int_rate', 'grade', 'sub_grade',
    'emp_length_years', 'emp_length_missing', 'home_ownership',
    'annual_inc', 'purpose', 'dti', 'dti_missing', 'delinq_2yrs',
    'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'revol_util_missing',
    'total_acc', 'issue_d', 'issue_year', 'issue_quarter',
    'addr_state', 'application_type', 'default_flag'
]

df_final = df[final_cols]
print("\nFinal shape:", df_final.shape)
print(df_final.dtypes)

# ── 11. Export clean CSV ───────────────────────────────────────────
df_final.to_csv('../data/loans_clean.csv', index=False)
print("\nSaved to data/loans_clean.csv")

Raw shape: (2260668, 20)
After filtering to resolved loans: (1303638, 20)

Target distribution:
default_flag
0    0.799265
1    0.200735
Name: proportion, dtype: float64
dti: 0.02% missing
revol_util: 0.06% missing

Dropped 0 rows with nulls in critical fields

Final shape: (1303638, 25)
loan_amnt                      int64
term                           int64
int_rate                     float64
grade                         object
sub_grade                     object
emp_length_years             float64
emp_length_missing             int64
home_ownership                object
annual_inc                   float64
purpose                       object
dti                          float64
dti_missing                    int64
delinq_2yrs                  float64
open_acc                     float64
pub_rec                      float64
revol_bal                      int64
revol_util                   float64
revol_util_missing             int64
total_acc                    float64
issue_d 